In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName('AzureDE-InterviewPrep')
    .master('local[*]')
    .config('spark.sql.shuffle.partitions', '4')
    .getOrCreate()
)
spark

In [ ]:
clicks = spark.createDataFrame(
    [
        ('u1', '2024-01-01 10:00:00', 'home'),
        ('u1', '2024-01-01 10:05:00', 'product'),
        ('u1', '2024-01-01 10:05:00', 'product'),
        ('u2', '2024-01-01 11:00:00', 'home'),
        ('u2', '2024-01-02 09:00:00', 'checkout'),
        ('u3', '2024-01-02 12:00:00', 'home'),
    ],
    ['user_id', 'event_ts', 'page'],
)
users = spark.createDataFrame(
    [('u1', 'IN'), ('u2', 'US'), ('u3', 'IN'), ('u4', 'UK')],
    ['user_id', 'country'],
)
clicks.show()

+-------+-------------------+--------+
|user_id|           event_ts|    page|
+-------+-------------------+--------+
|     u1|2024-01-01 10:00:00|    home|
|     u1|2024-01-01 10:05:00| product|
|     u1|2024-01-01 10:05:00| product|
|     u2|2024-01-01 11:00:00|    home|
|     u2|2024-01-02 09:00:00|checkout|
|     u3|2024-01-02 12:00:00|    home|
+-------+-------------------+--------+



# Q1. Deduplicate user+timestamp+page.

In [ ]:
clicks_dedup = clicks.dropDuplicates(
    ["user_id", "event_ts", "page"]
)

clicks_dedup.show()

+-------+-------------------+--------+
|user_id|           event_ts|    page|
+-------+-------------------+--------+
|     u1|2024-01-01 10:05:00| product|
|     u1|2024-01-01 10:00:00|    home|
|     u2|2024-01-01 11:00:00|    home|
|     u2|2024-01-02 09:00:00|checkout|
|     u3|2024-01-02 12:00:00|    home|
+-------+-------------------+--------+



# Q2. Last event per user.

In [ ]:
last_event_window = (
    Window
    .partitionBy("user_id")
    .orderBy(F.col("event_ts").desc())
)

In [ ]:
last_event_per_user = (
    clicks_dedup
    .withColumn(
        "rn",
        F.row_number().over(last_event_window)
    )
    .filter(F.col("rn") == 1)
    .drop("rn")
)
last_event_per_user.show()

+-------+-------------------+--------+
|user_id|           event_ts|    page|
+-------+-------------------+--------+
|     u1|2024-01-01 10:05:00| product|
|     u2|2024-01-02 09:00:00|checkout|
|     u3|2024-01-02 12:00:00|    home|
+-------+-------------------+--------+



# Q3. Distinct active users by country.

In [ ]:
active_users_by_country = (
    clicks_dedup
    .join(
        users,
        on="user_id",
        how="inner"
    )
    .groupBy("country")
    .agg(
        F.countDistinct("user_id").alias("active_users")
    )
    .orderBy("country")
)

active_users_by_country.show()

+-------+------------+
|country|active_users|
+-------+------------+
|     IN|           2|
|     US|           1|
+-------+------------+



# Q4. Users with zero events.

In [ ]:
users_zero_events = (
    users
    .join(
        clicks_dedup,
        on="user_id",
        how="left_anti"
    )
)

users_zero_events.show()

+-------+-------+
|user_id|country|
+-------+-------+
|     u4|     UK|
+-------+-------+



# Q5. Design bronze/silver/gold for clickstream on Azure.
                    CLICKSTREAM SOURCE
                           |
                           v
                 Azure Event Hubs / APIs
                           |
                           v
                    ┌─────────────┐
                    │   BRONZE    │
                    │ Raw Events  │
                    └──────┬──────┘
                           |
                    Clean / Validate
                    Deduplicate
                    Standardize
                           |
                           v
                    ┌─────────────┐
                    │   SILVER    │
                    │ Clean Events │
                    └──────┬──────┘
                           |
                    Business Logic
                    Aggregations
                           |
                           v
                    ┌─────────────┐
                    │    GOLD     │
                    │    KPIs     │
                    └──────┬──────┘
                           |
                           v
                       Power BI